In [ ]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
import time

# Specify the path to your Parquet file
FASSE_DIR = ""
ZIP2COUNTY_DIR = ""

# reading in crosswalk file from pipeline
DTYPE_DICT = {"county" : str,
             "zip": str}
crosswalk_file = pd.read_csv(f"{ZIP2COUNTY_DIR}/zip2county_master_xwalk_2010_2023_tot_ratio_one2one.csv", dtype=DTYPE_DICT)
unique_zip = set(crosswalk_file["zip"])

In [ ]:
# function that reads enrollment data from FASSE, returns data frame with bene_id and zip columns
def get_df(year = None):
    # if year is equal to None, download ALL years of enrollment files (requires lots of memory!)
    # it was previously "enroll_{year}.parquet
    if year == None:
        parquet_file_path = f"{FASSE_DIR}/mbsf_mortality_denom_*.parquet"
    else:
        parquet_file_path = f"{FASSE_DIR}/mbsf_mortality_denom_{year}.parquet"
    # Connect to an in-memory DuckDB instance
    con = duckdb.connect()
    # Query the Parquet file to select specific columns
    query = f"""
        SELECT 
            bene_id, 
            CAST(zip AS VARCHAR) AS zip
        FROM '{parquet_file_path}'
    """
    # Execute the query and fetch the results as a Pandas DataFrame
    df = con.execute(query).fetchdf()
    
    print("fetched df")
    # string formatting zip codes
    df["zip_str"] = df['zip'].astype(str).str.replace('.0', '', regex=False)
    df["zip_str"] = df["zip_str"].astype(str).str.zfill(5)
    return(df)

Now we read in the parquet file

In [ ]:
# looping through and reading parquet files
zip_counts = pd.DataFrame()
set_bene_id = set()
for year in range(2000, 2019):
    print(year)
    start_time = time.time()
    
    # read in cms beneficiary file
    df_tmp = get_df(year)
    set_bene_id.update(zip(df_tmp["bene_id"], df_tmp["zip_str"]))
    
    # summarizing number of beneficiaries in each zip
    zc_tmp = df_tmp.groupby('zip_str').size().reset_index(name='count')
    zc_tmp["year"] = year
    zip_counts = pd.concat([zip_counts, zc_tmp], axis=0)
    zip_counts.to_csv("zip_counts_out_newpath.csv")
    
    # print compute time
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Elapsed time: {elapsed_time:.2f} seconds")

In [ ]:
# Also option to load aggregated files if they are already in your directory
dtype_dict_zc={"zip_str": str}
zip_counts = pd.read_csv("zip_counts_out_newpath.csv", dtype=dtype_dict_zc, index_col=0)
# zip_counts = pd.concat([pd.read_csv("zip_counts_out.csv", dtype=dtype_dict_zc, index_col=0),
#                         pd.read_csv("zip_counts_out_2.csv", dtype=dtype_dict_zc, index_col=0)], axis=0)

zip_counts['zip_in_xwalk'] = zip_counts['zip_str'].isin(unique_zip)

In [ ]:
# Aggregating for final result of dataset coverage
result = zip_counts.groupby('year').apply(
    lambda x: pd.Series({
        'total_included': x.loc[x['zip_in_xwalk'], 'count'].sum(),
        'pct_included': x.loc[x['zip_in_xwalk'], 'count'].sum() / x['count'].sum()
    }),include_groups=True).reset_index()
result["year"] = result["year"].astype(int)

# Create the line plot
plt.figure(figsize=(10, 6))
plt.plot(result['year'], result['pct_included'], marker='o')
#plt.bar(result['year'], result['pct_included'])

# Add titles and labels
plt.title('Fraction of beneficiaries matched to county')
plt.xlabel('Year')
plt.ylabel('Fraction matched')
plt.ylim((0.8,1))
plt.xticks([2000, 2003, 2006, 2009, 2012, 2015, 2018])

# Display the plot
plt.grid(True)
plt.show()

The method we used above was generally accurate, but it was not "exactly" correct because it looked to see if a zip code was matched to a county code for any year, instead of just one year. In the next block, we re-do the above calculations but for a subset of the data (2010-2023) where we have exact matches from the HUD crosswalks. We can see the results are almost exactly the same. 

In [ ]:
bene_id_df = pd.DataFrame(list(set_bene_id), 
                          columns=["bene_id", "zip_str"])
bene_id_df['zip_in_xwalk'] = bene_id_df['zip_str'].isin(unique_zip)
# total fraction of beneficiaries that are included
print(sum(bene_id_df['zip_in_xwalk'])/len(bene_id_df))

In [ ]:
# the following looks at EXACT matches from 2010 to 2018, i.e. a zip matches to a county for a specific year 
zc_2010_23 = zip_counts[zip_counts["year"] >= 2010]
zc_2010_23 = pd.merge(zc_2010_23, crosswalk_file, how="left", left_on=["zip_str", "year"], right_on =["zip", "year"])

zc_2010_23.groupby('year').apply(
    lambda x: pd.Series({
        'total_included': x.loc[x['county'].notna(), 'count'].sum(),
        'pct_included': x.loc[x['county'].notna(), 'count'].sum() / x['count'].sum()
    }),
include_groups=True).reset_index()

As we can see, the true amount of missing beneficiaries is actually quite small!

Now we'll switch gears to look at PO boxes and beneficiaries. In order to do this, we'll pull in the uds_master_xwalk intermediate dataset which has PO box information. 

In [ ]:
# reading in uds zip --> zcta file
po_df = pd.read_csv("uds_clean_xwalk.csv", 
                    dtype={"zip": str, "zcta": str},
                    on_bad_lines='skip', 
                    sep=";", 
                    low_memory=False)
po_zips = set(po_df["zip"][po_df["is_post_office"]])

In [ ]:
# summary data frame that contains the percentage of beneficiaries that are in po boxes
po_box_dist = zip_counts.groupby('year').apply(
    lambda x: pd.Series({
        'total_pobox': x.loc[x['zip_str'].isin(po_zips), 'count'].sum(),
        'pct_pobox': x.loc[x['zip_str'].isin(po_zips), 'count'].sum() / x['count'].sum()
    }),
include_groups=True).reset_index()

# Create the line plot
plt.figure(figsize=(10, 6))
plt.plot(po_box_dist['year'], po_box_dist['pct_pobox'], marker='o')
#plt.bar(result['year'], result['pct_included'])

# Add titles and labels
plt.title('Fraction of Beneficiaries with P.O. Box ZIP Codes')
plt.xlabel('Year')
plt.ylabel('Fraction of Beneficiaries')
plt.xticks([2000, 2003, 2006, 2009, 2012, 2015, 2018])

# Display the plot
plt.grid(True)
plt.show()

The most peculiar thing here is the giant "bump" in beneficiaries listed under P.O. boxes in 2000-2003. This is so strange that it seems like it could be an artifact of something else going on in the data. Because we only have zip-->zcta data from 2009 onwards, it's possible that a zip code was listed as a P.O. box from 2009 onwards, but was actually a "normal" zip code 2000-2003. In any case the P.O. box status of a zip code is not critical to our analysis so this is not overly alarming.

In [ ]:
# filtering dataset so is exactly within range
zc_2009_2020 = zip_counts[(zip_counts["year"] >= 2009) & (zip_counts["year"] <= 2020)]
zc_2009_2020 = pd.merge(zc_2009_2020, po_df, 
                        how="left", 
                        left_on=["zip_str", "year"], 
                        right_on =["zip", "year"])
zc_2009_2020 = zc_2009_2020[zc_2009_2020['zcta'].notna()]
zc_2009_2020.groupby('year').agg(
    total_pobox=('count', lambda x: x[zc_2009_2020.loc[x.index, 'is_post_office']].sum()),
    pct_pobox=('count', lambda x: x[zc_2009_2020.loc[x.index, 'is_post_office']].sum() / x.sum())
).reset_index()